# Õppetund 18: AI agentide turvamine krüptograafiliste kviitungitega

## Praktikavihik

See vihik juhendab läbi nelja harjutuse:

1. **Allkirjasta oma esimene kviitung** agendi tööriistakutse jaoks ja kontrolli seda.
2. **Muuda kviitungit** ja vaata, kuidas kontroll ebaõnnestub.
3. **Ehitada kolme kviitungi ahel** ja kinnitada ahela terviklikkus.
4. **Mähkida Microsoft Agent Frameworki tööriistakutse** nii, et iga tegevus tekitab kviitungi.

Kõik krüptograafilised primitiivid tuuakse sisse hästi hooldatud teekidest (`pynacl` Ed25519 jaoks, `jcs` RFC 8785 kanonilise JSONi jaoks, `hashlib` Pythoni standardteegist SHA-256 jaoks). Kviitungi loogika ise on lihtne Python, mida saad lugeda ja muuta.

Käivita lahtrid järjest. Iga osa on lühike ja iseseisev.


## Seadistamine

Paigaldage kaks sõltuvust. Mõlemal on leebed litsentsid (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Abitööriistad

Need kaks abiakut tegelevad base64url kodeerimisega (ilma täitmiseta) ja suvaliste objektide SHA-256 räsi genereerimisega. Need hoiavad ülejäänud märkmiku keskendunud kviitungi loogikale.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Jaotis 1: Allkirjasta oma esimene kviitung

Kujutame ette, et meie agent ettevõttest **Contoso Travel** on just otsinud kliendi jaoks lende Sydney'st Los Angelesse. Me tahame selle tööriistakõne salvestada allkirjastatud kviitungina, et tulevane audiitor saaks seda kinnitada ilma, et ta peaks meile usaldama.

### Samm 1.1: Loo allkirjastamisvõti

Tootmises elaks agendi allkirjastamisvõti riistvaralist turvamoodulis (HSM), Azure Key Vault'is või sarnases kaitstud hoidlas. Selleks õppetunniks genereerime uue võtme mälus.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### 1.2 samm: Koosta kviitungi sisu

Sisu sisaldab kõike, mida soovime, et kviitung kinnitaks: kes tegutses, millist tööriista kasutati, milliste argumentidega, mis tagastati, millise poliisi alusel ja millal. Me räsiame argumendid ja tulemuse asemel, et lisada need otse, et kviitung ei lekiks tundlikku teavet.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Samm 1.3: Allkirjasta ja koosta kviitung

Kolm sammu:

1. Kanoniseeri laad JCS-i abil, nii et kaks rakendust, mis toodavad sama loogilist kviitungit, toodavad baitidest identsed baitid.
2. Räsi kanoniseeritud baitid SHA-256-ga.
3. Allkirjasta räsi Ed25519 privaatvõtmega.

Allkiri lisatakse seejärel originaallaadile, et saada lõplik kviitung.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Samm 1.4: Kviitungi kontrollimine

Kontrollimine teeb protsessi vastupidiseks. Eemaldame allkirja, arvutame uuesti kanonilise räsi ja kontrollime allkirja kviitungis oleva avaliku võtmega.

Audiitor, kes teeb selle kontrolli, ei vaja meist muud kui vaid kviitungit ennast. Pole vaja mingit teenust kutsuda, võtmekataloogi pärida ega usaldust nõuda.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

Peaksid nägema `Kviitung on kehtiv: True`. Agent on loonud oma esimese krüptograafiliselt allkirjastatud auditi kirje.


## 2. peatükk: Kviitungi muutmine

Kviitungite mõte on see, et neid ei saa muuta märkamatult. Tõestame seda.

Muudame täpselt ühe tähemärgi kviitungis ja vaatame, kuidas kontroll läbi kukub.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### Mis just juhtus?

Kui me muutsime `policy_id`, muutusid ka kanoonilised baidid. Nende baitide SHA-256 räsi muutus. Allkiri (mis oli algse räsi üle) ei vasta enam uuele räsi väärtusele. Verifitseerimine tagastab õigesti `False`.

Pole mingit võimalust kviitungi ühtegi välja muuta ja ikkagi verifitseerimiselt läbi pääseda, kui ründajal ei ole privaatvõtit. Seni, kuni privaatvõti on võtmehoidlas ja avalik võti on avaldatud, on võltsimise varjamine võimatu.

Proovi ise: muuda ülaltoodud lahtris `tool_name` või `agent_id` või `timestamp` ja käivita uuesti. Iga muutus toodab kehtetu kviitungi.


## 3. jaotis: Kviitungid omavahel ahelana ühendada

Üksainus kviitung kaitseb üht toimingut. Enamik agente sooritab palju toiminguid. Selleks, et kogu järjestus oleks näha rikkumisekindel, ühendame iga kviitungi eelmisega, lisades uue kviitungi andmetesse eelmise kviitungi räsi.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Kui keegi eemaldab või ümber korraldab kviitungi, katkeb ahel täpselt sellel kohal. Iga hilisema kviitungi kontroll ebaõnnestub, sest selle `previous_receipt_hash` ei lange enam kokku selle eelkäija tegeliku räsidega.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Nüüd murra ahel, manipuleerides keskmise kviitungiga ja kinnita uuesti. Manipuleeritud kviitung ebaõnnestub oma allkirja kontrollimisel, JA järgmine kviitung ebaõnnestub oma ahela lingi kontrollimisel (sest selle `previous_receipt_hash` ei vasta enam muudetud keskmise kviitungi räsidele).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Kviitung 0 kehtib endiselt (see ei olnud muudetud ja ei sõltu eellane). Kviitung 1 ei läbiks allkirjakontrolli, sest me muutsime `tool_args_hash`. Kviitung 2 ei passi kettaseose kontrolli, sest tema `previous_receipt_hash` arvutati algse (nüüd muudetud) kviitungi 1 vastu.

Isegi kui ründaja üritaks muudetud kviitungit 1 uuesti allkirjastada (mida ta ei saa teha ilma privaatvõtmeta), avaldaks kviitungi 2 kettaseose vastuolu ikka moonutuse. Muudatuse varjamiseks peaks ründaja uuesti allkirjastama iga kviitungi alates muutmispunktist, mis nõuab privaatvõtme omamist.


## 4. jaotis: agente tööriista kutsumine koos kviitungi allkirjastamisega

Tõelises juurutamises ei taha sa, et iga agente kirjutaja peaks meeles pidama `make_receipt` kutsumist. Sa tahad, et kviitungi allkirjastamine oleks automaatne iga tööriista kutse puhul.

Siin on lihtsaim muster: wrapper-klass, mis võtab mis tahes kutsutava tööriistafunktsiooni ja tagastab sellest kviitungi väljastava versiooni. See kohandub mis tahes agente raamistikuga, sealhulgas Microsoft Agent Frameworkiga (`agent_framework.foundry`).

Kui sul pole Microsoft Foundry projekti loodud, siis ka kohaliku mocki näide allpool demonstreerib seda mustrit.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


### Integreerimine Microsoft Agent Frameworkiga

Ülaltoodud `ReceiptedTool` mähis ei sõltu raamistiku tüüpidest. Selle kasutamiseks Microsoft Agent Frameworki abil loodud agendi sees registreerige mähitud funktsioon tööriistana. Kontuur (te asendaksite moki päris Microsoft Foundry tööriistaregistratsiooniga):

```python
# Pseudokood, mis näitab integratsiooni kuju.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="Oled Contoso Travel agent ...",
#     tools=[receipted_lookup],   # ümbritsetud tööriist, mitte algne funktsioon
# )
# response = agent.run("Leia lennud Sydneyst Los Angelesse juunis.")
#
# # Pärast käivitamist on iga tööriistakutse juures agent kirjutatud kviitung:
# audit_chain = receipted_lookup.receipts
```

Agendi raamistik ei pea teadma midagi kviitungitest. Kviitungu allkirjastamine on mähitud tööriista ümber, mitte raamistikku sisse ehitatud. Nii lisate päritolu olemasolevasse agendi koodi ilma agenti ümber kirjutamata.


## Kordus-kokkuvõte ja lisaväljakutse

Sul on:

- Genereeritud Ed25519 võtit.
- Koostanud ja allkirjastanud kviitungi agendi tööriista kõne jaoks.
- Kontrollinud kviitungit võrguühenduseta, kasutades ainult avalikku võtit.
- Muudetud kviitungit ja täheldatud, et kontroll ebaõnnestub.
- Koostanud kolme kviitungi hash-ahelaga jada.
- Muudetud ahelas keskmist elementi ja täheldatud nii allkirja- kui ka ahela-viga.
- Mähkinud tööriista funktsiooni automaatseks kviitungite allkirjastamiseks.

**Lisaväljakutse.** Täienda kviitungi skeemi väljal `request_id` (UUID hajutatud jälgimiseks). Uuenda `make_receipt` seda kaasama ning kinnita, et kviitungid kontrollivad endiselt täielikult. Seejärel muuda seda välja pärast allkirjastamist ja kinnita, et kontroll ebaõnnestub. See sunnib sind mõistma, kuidas iga bait kanonilises kodeeringus allkirja mõjutab.

**Oluline piir.** Kviitungid tõendavad kolme asja ja ainult kolme asja: atribuutimist (see võti allkirjastas selle sisu), terviklikkust (sisu pole pärast allkirjastamist muutunud) ja järjekorda (see kviitung tuli pärast seda kviitungit). Need EI tõenda, et agendi tegevus oli õige, et `policy_id` all nimetatud poliitikat tegelikult hinnati või et agent järgis kõiki reegleid. Kviitungid on alus. Halduse ehitad peale seda süsteemi.

Loe õpetuse README uuesti, hoides seda piiri meeles. Kõige tavalisem viga, mida meeskonnad kviitungitega teevad, on eelduse, et "meil on kviitungid" tähendab "meid juhitakse." See ei pea paika. Kviitungid teevad agendi käitumise auditeeritavaks. Need ei tee seda õigeks.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades AI tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi me püüdleme täpsuse poole, palun pange tähele, et automatiseeritud tõlgetes võib esineda vigu või ebatäpsusi. Originaaldokument selle emakeeles tuleks pidada autoriteetseks allikaks. Olulise teabe puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlkega seotud eksimustest või valesti mõistmistest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
